# mpH2MM / smFRET Analysis — PTU files (Alexa555 / Alexa647)

**Donor:** Alexa 555 (excitation ~520 nm, emission ~570 nm)  
**Acceptor:** Alexa 647 (excitation ~640 nm, emission ~690 nm)  

**Pipeline:**
1. Convert `.ptu` → Photon-HDF5 via `phconvert`
2. Filter junk detectors (keep det 0 and 1 only)
3. Burst search + stoichiometry filter (S filter)
4. First-round H²MM — identify donor-only bursts
5. Second-round H²MM on FRET-active bursts
6. Dwell-time extraction and survival plots
7. BVA (Burst Variance Analysis)

**Environment:** Python 3.10, numpy 1.26.4, fretbursts 0.8.3, burstH2MM 0.1.7, H2MM_C 2.1.2, phconvert 0.10.1

## 1 · Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import h5py
import shutil

import fretbursts as frb
import phconvert as phc
import burstH2MM as bhm
import H2MM_C as h2

print('NumPy      :', np.__version__)
print('FRETbursts :', frb.__version__)
print('phconvert  :', phc.__version__)
print('burstH2MM  :', bhm.__version__)
print('H2MM_C     :', h2.__version__)

## 2 · Helper functions

In [ ]:
def ptu_to_hdf5(
    ptu_file: str,
    output_path: str,
    description: str = 'smFRET measurement — Alexa555 / Alexa647',
    author: str = '',
    sample_name: str = '',
    buffer_name: str = '',
    dye_names: str = 'Alexa555, Alexa647',
):
    """
    Convert a PicoQuant .ptu file to Photon-HDF5 format.

    Notes
    -----
    - Uses phconvert.pqreader.load_ptu (5-value return: timestamps,
      detectors, nanotimes, meta_dict, laser_array)
    - measurement_type = smFRET-usALEX (CW lasers, microsecond alternation)
    - lifetime=False: nanotimes not stored (incompatible with CW usALEX)
    - donor   detector = 1 (Alexa555 emission)
    - acceptor detector = 0 (Alexa647 emission)
    - ALEX periods: donor=(0,1000), acceptor=(1000,2000) ticks
    """
    print(f'Reading {ptu_file} ...')
    timestamps, detectors, nanotimes, meta, laser_array = phc.pqreader.load_ptu(ptu_file)

    timestamps_unit = meta['timestamps_unit']
    tcspc_num_bins  = int(nanotimes.max()) + 1
    laser_rep_rate  = float(laser_array[0]) if len(laser_array) > 0 else 1.0 / timestamps_unit
    tcspc_unit      = 1.0 / (laser_rep_rate * tcspc_num_bins)

    # Diagnostic printout
    unique_det, counts = np.unique(detectors, return_counts=True)
    print('Detectors found     :', dict(zip(unique_det.tolist(), counts.tolist())))
    print('timestamps_unit     :', timestamps_unit, 's')
    print('laser_rep_rate      :', laser_rep_rate, 'Hz')
    print('tcspc_num_bins      :', tcspc_num_bins)
    print('acquisition_duration:', meta['acquisition_duration'], 's')

    # TCSPC decay plot — use to verify ALEX period windows
    fig, ax = plt.subplots(figsize=(8, 3))
    for d in unique_det:
        mask = detectors == d
        ax.hist(nanotimes[mask], bins=256, histtype='step', label=f'det {d}')
    ax.set_yscale('log')
    ax.set_xlabel('nanotime (ticks)')
    ax.set_ylabel('counts')
    ax.legend()
    ax.set_title('TCSPC decays by detector — verify ALEX period windows here')
    plt.tight_layout()
    plt.show()

    # Photon-HDF5 metadata
    # smFRET-usALEX: CW lasers alternated on microsecond timescale
    # excitation_cw=[True,True] required; no laser_repetition_rate in measurement_specs
    # lifetime=False: mandatory when excitation_cw=True
    measurement_specs = dict(
        measurement_type='smFRET-usALEX',
        alex_period=2000,
        alex_excitation_period1=(0, 1000),
        alex_excitation_period2=(1000, 2000),
        detectors_specs=dict(spectral_ch1=[1], spectral_ch2=[0]),
    )
    setup = dict(
        num_pixels=2,
        num_spots=1,
        num_spectral_ch=2,
        num_polarization_ch=1,
        num_split_ch=1,
        modulated_excitation=True,
        excitation_alternated=[True, True],
        excitation_cw=[True, True],
        lifetime=False,
        laser_repetition_rates=[laser_rep_rate],
        excitation_wavelengths=[520e-9, 640e-9],
        detection_wavelengths=[570e-9, 690e-9],
    )
    photon_data = dict(
        timestamps=timestamps,
        timestamps_specs={'timestamps_unit': timestamps_unit},
        detectors=detectors,
        measurement_specs=measurement_specs,
    )
    h5_data = dict(
        identity=dict(author=author, author_affiliation=''),
        sample=dict(sample_name=sample_name, buffer_name=buffer_name, dye_names=dye_names),
        provenance=dict(filename=ptu_file, creation_time=str(meta.get('creation_time', ''))),
        description=description,
        photon_data=photon_data,
        setup=setup,
    )
    phc.hdf5.save_photon_hdf5(h5_data, h5_fname=output_path, overwrite=True, close=True)
    print(f'Saved Photon-HDF5 → {output_path}')

In [ ]:
def filter_detectors(input_h5: str, output_h5: str, keep: tuple = (0, 1)):
    """
    Copy HDF5 keeping only photons from detectors in `keep`.
    Removes junk channels (e.g. det 68, 127 sync/IRF artifacts).
    """
    shutil.copy(input_h5, output_h5)
    with h5py.File(output_h5, 'r+') as f:
        det = f['photon_data/detectors'][:]
        ts  = f['photon_data/timestamps'][:]
        mask = np.zeros(len(det), dtype=bool)
        for d in keep:
            mask |= (det == d)
        print(f'Keeping {mask.sum()} of {len(mask)} photons '
              f'({mask.sum()/len(mask)*100:.1f}%) — detectors {keep}')
        for key, arr in [('detectors', det), ('timestamps', ts)]:
            del f[f'photon_data/{key}']
            f.create_dataset(f'photon_data/{key}', data=arr[mask])
    print(f'Filtered HDF5 saved → {output_h5}')

In [ ]:
def load_and_burst_search(
    h5_file: str,
    D_ON: tuple = (0, 1000),
    A_ON: tuple = (1000, 2000),
    m: int = 10,
    F: float = 6.0,
    min_burst_size: int = 30,
    min_naa: int = 20,
):
    """
    Load Photon-HDF5, apply ALEX periods, run burst search and selection.

    Parameters
    ----------
    D_ON / A_ON      : ALEX excitation windows in timestamp ticks
    m                : min consecutive photons for burst start
    F                : burst threshold (x background rate)
    min_burst_size   : min Dex photons per burst
    min_naa          : min Aex-Aem photons per burst

    Returns
    -------
    data     : all bursts (FRETbursts Data object)
    data_sel : burst-selected Data object
    """
    data = frb.loader.photon_hdf5(h5_file)
    data.add(
        donor=1,           # Alexa555 emission → detector 1
        acceptor=0,        # Alexa647 emission → detector 0
        alex_period_donor=D_ON,
        alex_period_acceptor=A_ON,
    )
    frb.loader.alex_apply_period(data)
    data.calc_bg(fun=frb.bg.exp_fit, time_s=30, F_bg=1.7)
    data.burst_search(m=m, F=F)
    data.fuse_bursts(ms=0)
    data_sel = data.select_bursts(frb.select_bursts.size, add_naa=False, th1=min_burst_size)
    data_sel = data_sel.select_bursts(frb.select_bursts.naa, th1=min_naa)
    return data, data_sel

In [ ]:
# BVA helpers
def BVA(data, chunk_size: int = 5):
    """Burst Variance Analysis — compute sub-burst FRET std-dev per burst."""
    E_eff, std_E = [], []
    for ich, mburst in enumerate(data.mburst):
        stdE, E = [], []
        Aem = data.get_ph_mask(ich=ich, ph_sel=frb.Ph_sel(Dex='Aem'))
        Dex = data.get_ph_mask(ich=ich, ph_sel=frb.Ph_sel(Dex='DAem'))
        for istart, istop in zip(mburst.istart, mburst.istop):
            phots = Aem[istart:istop+1][Dex[istart:istop+1]]
            Esub  = [phots[nb:ne].sum()
                     for nb, ne in zip(range(0, phots.size, chunk_size),
                                       range(chunk_size, phots.size+1, chunk_size))]
            stdE.append(np.std(Esub) / chunk_size)
            E.append(sum(Esub) / (len(Esub) * chunk_size) if Esub else np.nan)
        E_eff.append(np.array(E))
        std_E.append(np.array(stdE))
    return E_eff, std_E

def bin_bva(E, std, R: int = 10, B_thr: int = 50):
    """Bin BVA results for a smoother plot."""
    E_cat   = np.concatenate(E)
    std_cat = np.concatenate(std)
    bn      = np.linspace(0, 1, R + 1)
    std_avg, E_avg = np.empty(R), np.empty(R)
    for i, (bb, be) in enumerate(zip(bn[:-1], bn[1:])):
        mask = (bb <= E_cat) & (E_cat < be)
        if mask.sum() > B_thr:
            std_avg[i] = np.mean(std_cat[mask])
            E_avg[i]   = np.mean(E_cat[mask])
        else:
            std_avg[i] = E_avg[i] = -1
    return E_avg, std_avg

# Shot-noise reference curve
n_bva = 5
x_T   = np.arange(0, 1.01, 0.01)
y_T   = np.sqrt((x_T * (1 - x_T)) / n_bva)

## 3 · Convert PTU → Photon-HDF5

Set `ptu_file` to your measurement. The TCSPC histogram printed below
lets you verify that `alex_period_donor=(0,1000)` and
`alex_period_acceptor=(1000,2000)` match the actual excitation windows.

In [ ]:
# ── USER INPUT ────────────────────────────────────────────────────────────
ptu_file    = r'C:/path/to/your_measurement.ptu'   # ← change this
output_h5   = 'measurement.h5'                     # ← raw HDF5 output
filtered_h5 = 'measurement_filtered.h5'            # ← filtered HDF5
# ─────────────────────────────────────────────────────────────────────────

ptu_to_hdf5(
    ptu_file=ptu_file,
    output_path=output_h5,
    description='smFRET measurement — Alexa555 / Alexa647',
    author='',           # ← your name
    sample_name='',      # ← e.g. 'APO', 'HOLO'
    buffer_name='',      # ← e.g. 'Tris pH 8.0'
)

## 4 · Filter junk detectors

Removes detector 68 (noise) and detector 127 (laser sync/IRF artifact).
Only photons from det 0 (Alexa647) and det 1 (Alexa555) are kept.

In [ ]:
filter_detectors(output_h5, filtered_h5, keep=(0, 1))

## 5 · Burst search + stoichiometry filter

Adjust `m`, `F`, `min_burst_size`, `min_naa` if too few or too many bursts.
S filter (0.25–0.75) removes donor-only and acceptor-only bursts.

In [ ]:
smfret_raw, smfret_sel = load_and_burst_search(
    h5_file=filtered_h5,
    D_ON=(0, 1000),        # donor excitation window (ticks)
    A_ON=(1000, 2000),     # acceptor excitation window (ticks)
    m=10,                  # min consecutive photons for burst start
    F=6.0,                 # burst threshold (x background rate)
    min_burst_size=30,     # min Dex photons per burst
    min_naa=20,            # min Aex-Aem photons per burst
)
print(f'Total bursts          : {smfret_raw.mburst[0].num_bursts}')
print(f'Bursts after selection: {smfret_sel.mburst[0].num_bursts}')

# Stoichiometry filter — keep only genuine FRET bursts (0.25 < S < 0.75)
smfret_s = smfret_sel.select_bursts(frb.select_bursts.S, S1=0.25, S2=0.75)
print(f'Bursts after S filter : {smfret_s.mburst[0].num_bursts}')

frb.alex_jointplot(smfret_s)
plt.title('After burst search + S filter')
plt.show()

## 6 · First-round H²MM + donor-only filtering

Fits 1–6 state models and uses ICL/BICp to select the optimal number.
Then identifies and removes any remaining donor-only bursts.

**Model index is 0-based:** `models[0]`=1-state, `models[1]`=2-state, etc.
Set `round1_idx` to match the ICL/BICp suggestion.

In [ ]:
# Fit models
print('Fitting first-round H2MM models...')
smfret_mp = bhm.BurstData(smfret_s)
smfret_mp.models.calc_models(max_state=6, max_iter=3600)
print('Done.')

# Model selection
bhm.ICL_plot(smfret_mp.models)
plt.title('ICL — first round')
plt.show()

bhm.BICp_plot(smfret_mp.models)
plt.axhline(0.005, color='red', linestyle='--', label='threshold 0.005')
plt.legend()
plt.title('BICp — first round')
plt.show()

In [ ]:
# ── SET based on ICL/BICp plots above ─────────────────────────────────────
# Pick first model where BICp drops below 0.005
# 0-indexed: models[0]=1-state, models[1]=2-state, models[2]=3-state ...
round1_idx = 3   # ← adjust if needed (3 = 4-state model)
# ─────────────────────────────────────────────────────────────────────────

model_r1 = smfret_mp.models[round1_idx]
print(f'Model states : {model_r1.nstate}')
print(f'E per state  : {model_r1.E}')
print(f'S per state  : {model_r1.S}')

bhm.dwell_ES_scatter(model_r1, alpha=0.2, s=2)
bhm.trans_arrow_ES(model_r1)
plt.title(f'First-round H2MM — {model_r1.nstate} states')
plt.show()

# Identify donor-only state (highest S value)
donor_only_state = int(np.argmax(model_r1.S))
print(f'\nDonor-only state: {donor_only_state}  '
      f'E={model_r1.E[donor_only_state]:.2f}  '
      f'S={model_r1.S[donor_only_state]:.2f}')

# Keep bursts with at least one photon outside donor-only state
fret_mask = np.array([
    not np.all(path == donor_only_state)
    for path in model_r1.path
])
print(f'Bursts before H2MM filter : {len(fret_mask)}')
print(f'Bursts after  H2MM filter : {fret_mask.sum()}')

# smfret_fret — FRET-active dataset used for all downstream analysis
smfret_fret = smfret_s.select_bursts_mask_apply(np.where(fret_mask))

frb.alex_jointplot(smfret_fret)
plt.title('FRET-active bursts after first-round H2MM filter')
plt.show()

## 7 · Second-round H²MM (on FRET-active bursts)

Set `final_idx` to the model suggested by the second-round BICp plot
(first model where BICp drops below 0.005).

In [ ]:
# Fit models on FRET-active bursts
print('Fitting second-round H2MM models...')
smfret_fret_mp = bhm.BurstData(smfret_fret)
smfret_fret_mp.models.calc_models(max_state=6, max_iter=3600)
print('Done.')

bhm.ICL_plot(smfret_fret_mp.models)
plt.title('ICL — second round')
plt.show()

bhm.BICp_plot(smfret_fret_mp.models)
plt.axhline(0.005, color='red', linestyle='--', label='threshold 0.005')
plt.legend()
plt.title('BICp — second round')
plt.show()

In [ ]:
# ── SET based on second-round BICp plot ───────────────────────────────────
final_idx = 4   # ← adjust (4 = 5-state model, 0-indexed)
# ─────────────────────────────────────────────────────────────────────────

model_final = smfret_fret_mp.models[final_idx]
print(f'Final model states : {model_final.nstate}')
print(f'E per state        : {model_final.E}')
print(f'S per state        : {model_final.S}')
print(f'Trans matrix       :\n{model_final.trans}')

fig, ax = plt.subplots(figsize=(5, 4))
bhm.dwell_ES_scatter(model_final, alpha=0.6, s=1)
bhm.trans_arrow_ES(model_final, min_rate=0)
bhm.scatter_ES(model_final, s=100, c='r')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_xlabel(r'$E$'); ax.set_ylabel(r'$S$')
plt.title(f'Final H2MM — {model_final.nstate} states')
plt.tight_layout()
plt.show()

## 8 · Dwell-time extraction

In [ ]:
n_states = model_final.nstate
print(f'Total dwells        : {len(model_final.dwell_dur)}')
print(f'Dwell states present: {np.unique(model_final.dwell_state)}')

dwell_per_state = [
    model_final.dwell_dur[model_final.dwell_state == s]
    for s in range(n_states)
]

# Convert photons → ms using mean in-burst photon rate
bursts       = smfret_fret.mburst[0]
mean_ph_rate = bursts.counts.sum() / (bursts.width.sum() * smfret_fret.clk_p)
print(f'Mean in-burst rate  : {mean_ph_rate:.0f} ph/s')

print()
for s in range(n_states):
    dt_ms = dwell_per_state[s] / mean_ph_rate * 1000
    print(f'State {s} (E≈{model_final.E[s]:.2f}): '
          f'{len(dwell_per_state[s])} dwells  '
          f'mean={dt_ms.mean():.3f} ms  '
          f'max={dt_ms.max():.3f} ms')

# Survival function plots
fig, axes = plt.subplots(1, n_states, figsize=(4*n_states, 4), sharey=True)
if n_states == 1:
    axes = [axes]

for s, ax in enumerate(axes):
    dt    = dwell_per_state[s]
    dt    = dt[dt > 0]
    t_ms  = np.sort(dt / mean_ph_rate * 1000)
    surv  = 1 - np.arange(1, len(t_ms)+1) / len(t_ms)
    ax.semilogy(t_ms, surv, '.', markersize=3)
    ax.set_xlabel('Dwell time (ms)')
    ax.set_ylabel('Survival probability')
    ax.set_title(f'State {s}  E≈{model_final.E[s]:.2f}  (n={len(dt)})')

plt.suptitle('Dwell-time survival functions per H²MM state')
plt.tight_layout()
plt.show()

## 9 · BVA (Burst Variance Analysis)

Points above the shot-noise line (dashed) confirm genuine conformational
dynamics beyond photon counting noise.

In [ ]:
E_bva, std_bva = BVA(smfret_fret, chunk_size=n_bva)
E_avg, std_avg = bin_bva(E_bva, std_bva, R=10, B_thr=50)

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(np.concatenate(E_bva), np.concatenate(std_bva),
           s=2, alpha=0.3, color='grey', label='per burst')
valid = E_avg > 0
ax.scatter(E_avg[valid], std_avg[valid], s=40, color='blue',
           zorder=5, label='binned avg')
ax.plot(x_T, y_T, 'k--', label='shot noise')
ax.set_xlabel(r'$\langle E \rangle$')
ax.set_ylabel(r'$\sigma_E$')
ax.set_title('BVA — dynamics confirmed if points above dashed line')
ax.legend()
plt.tight_layout()
plt.show()